# Manantial Chatbot - RA2 + RA3
## Agentes Inteligentes + Observabilidad en Producción

Demostración completa del chatbot multi-agente con:
- **RA2**: Agentes especializados (Coordinator, Sales, Support)
- **RA3**: Trazabilidad, métricas y observabilidad

In [ ]:
# Setup
import sys
sys.path.insert(0, '/workspace')  # Ajustar según tu entorno

from main import chatbot
import json
from datetime import datetime

## Escenario 1: Cliente pregunta por precio
Flujo: Coordinator → SalesAgent → get_product_price()

In [ ]:
# Escenario 1: Consulta de precio
resultado1 = chatbot.procesar_mensaje(
    customer_name="Juan González",
    mensaje="Hola, ¿cuál es el precio del bidón de 20 litros?"
)

print("🤖 Respuesta:", resultado1['respuesta'])
print(f"⏱️  Tiempo: {resultado1.get('tiempo_ms', 0):.0f}ms")
print(f"📍 Trace ID: {resultado1['trace_id']}")
print(f"👤 Agente: {resultado1.get('agente', 'unknown')}")

## Escenario 2: Cliente quiere hacer un pedido
Flujo: Coordinator → SalesAgent → check_availability() → process_order()

In [ ]:
# Escenario 2: Hacer un pedido
resultado2 = chatbot.procesar_mensaje(
    customer_name="María López",
    mensaje="Quiero comprar 3 bidones de 12 litros para la zona sur. ¿Cuánto es y cuándo me llega?"
)

print("🤖 Respuesta:", resultado2['respuesta'])
print(f"⏱️  Tiempo: {resultado2.get('tiempo_ms', 0):.0f}ms")
print(f"📍 Trace ID: {resultado2['trace_id']}")

## Escenario 3: Cliente tiene una queja
Flujo: Coordinator → SupportAgent → create_support_ticket()

In [ ]:
# Escenario 3: Queja del cliente
resultado3 = chatbot.procesar_mensaje(
    customer_name="Juan González",
    mensaje="Mi pedido anterior llegó con 2 días de retraso. Esto no es aceptable."
)

print("🤖 Respuesta:", resultado3['respuesta'])
print(f"⏱️  Tiempo: {resultado3.get('tiempo_ms', 0):.0f}ms")
print(f"👤 Agente: {resultado3.get('agente', 'unknown')}")

## RA3: Observabilidad - Traces
Visualizar trazas completas de cada conversación

In [ ]:
# Inspeccionar traza de Escenario 1
trace_id = resultado1['trace_id']
traza = chatbot.sistema_traces.get_traza(trace_id)

if traza:
    print(f"📍 Trace ID: {traza.trace_id}")
    print(f"👤 Cliente: {traza.customer_name}")
    print(f"📝 Input: {traza.mensaje_entrada}")
    print(f"🤖 Output: {traza.respuesta_final[:100]}...")
    print(f"⏱️  Duración: {traza.duracion_total_ms:.0f}ms")
    print(f"✅ Exitoso: {traza.exitoso}")
    print(f"\n📊 Eventos en la traza:")
    for i, evento in enumerate(traza.eventos, 1):
        print(f"  {i}. {evento.nombre} ({evento.duracion_ms:.0f}ms) - {'✓' if evento.exitoso else '✗'}")

## RA3: Métricas Agregadas
Resumen de desempeño del sistema

In [ ]:
# Obtener métricas
metricas = chatbot.obtener_metricas()

print("📊 RESUMEN GENERAL")
print("="*50)
resumen = metricas['resumen_general']
print(f"Total interacciones: {resumen['total_interacciones']}")
print(f"Tasa de éxito: {resumen['tasa_exito']*100:.1f}%")
print(f"Latencia promedio: {resumen['promedio_latencia_ms']:.0f}ms")
print(f"Latencia mínima: {resumen['latencia_min_ms']:.0f}ms")
print(f"Latencia máxima: {resumen['latencia_max_ms']:.0f}ms")
print(f"Total tokens: {resumen['total_tokens']}")
print(f"Costo estimado: ${resumen['costo_estimado_usd']:.4f}")

print("\n👥 POR AGENTE")
print("="*50)
for agente, stats in metricas['resumen_por_agente'].items():
    print(f"\n{agente}:")
    print(f"  Total: {stats['total']}")
    print(f"  Exitosas: {stats['exitosas']}")
    print(f"  Tasa éxito: {stats['tasa_exito']*100:.1f}%")
    print(f"  Latencia: {stats['latencia_promedio_ms']:.0f}ms")

print(f"\n👤 CLIENTES ACTIVOS: {len(metricas['clientes_activos'])}")
for cliente in metricas['clientes_activos']:
    print(f"  - {cliente}")

## RA2: Persistencia - Historial de Cliente
Recuperar conversaciones anteriores del cliente

In [ ]:
# Obtener historial completo de Juan
historial = chatbot.obtener_historial_cliente("Juan González")

print(f"👤 Cliente: {historial['customer_name']}")
print(f"📊 Total sesiones: {historial['total_sesiones']}")
print(f"🛒 Total pedidos: {historial['total_pedidos']}")

if historial['sesiones']:
    print(f"\n📝 Sesiones:")
    for i, sesion in enumerate(historial['sesiones'][:3], 1):  # Mostrar primeras 3
        print(f"  {i}. {sesion['conversation_id']} ({sesion['created_at'][:10]})")
        print(f"     Mensajes: {len(sesion['messages'])}")

if historial['pedidos']:
    print(f"\n🛒 Pedidos:")
    for i, pedido in enumerate(historial['pedidos'][:3], 1):
        print(f"  {i}. {pedido['order_id']}: {pedido['quantity']}x {pedido['product']} (${pedido['total_price']})")

## Comparativa: Antes (RA1) vs Después (RA1+RA2+RA3)

### Antes (RA1 - Chatbot simple):
- ❌ Un solo agente monolítico
- ❌ Sin especialización (Sales/Support)
- ❌ Métricas limitadas
- ❌ Sin trazabilidad
- ❌ Memoria solo en RAM

### Después (RA1+RA2+RA3 - Sistema completo):
- ✅ 3 agentes especializados (Coordinator, Sales, Support)
- ✅ Enrutamiento inteligente de intents
- ✅ 9 herramientas explícitas reutilizables
- ✅ Trace IDs para debugging y auditoría
- ✅ BD persistente (SQLite)
- ✅ Métricas de desempeño (latencia, tokens, costo)
- ✅ Integración LangSmith lista
- ✅ Escalable a producción

## Próximos Pasos

1. **Integración LangSmith**: Activar LANGSMITH_TRACING en .env para dashboard visual
2. **Interfaz Web**: Streamlit + FastAPI para deployment
3. **Escalabilidad**: Redis para estado distribuido, PostgreSQL para persistencia
4. **Feedback Loop**: Recolectar satisfacción del cliente
5. **A/B Testing**: Comparar diferentes prompts y estrategias